# Assignment 3 — PASCAL VOC 2007 Object Detection

**Students**
- Dawod Ghifari — 520140154
- Hilal Chamtie — 540749283
- Darin Li — 500048292
- Akshar Hossain — 540823996


In this project, we use the PASCAL VOC 2007 dataset for **object detection**.

The dataset has been divided into training, validation, and test sets using the official VOC split files.

Students should use:
- train.txt for training
- val.txt for validation
- test.txt for final testing



The images are stored in JPEGImages.

The object detection annotations are stored in Annotations.

Each annotation file is an XML file that contains object class names and bounding-box coordinates.

**The code below prepares the dataset for this assignment.**

Please run the following code cells before starting your implementation.

### **1. Download and extract the dataset**


In [ ]:
!wget -q http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtrainval_06-Nov-2007.tar
!wget -q http://host.robots.ox.ac.uk/pascal/VOC/voc2007/VOCtest_06-Nov-2007.tar

# Extract files
!tar -xf VOCtrainval_06-Nov-2007.tar
!tar -xf VOCtest_06-Nov-2007.tar

### **2. Check the dataset structure**

In [ ]:
# Check the main VOC2007 folder

!ls VOCdevkit/VOC2007

##### **For this object detection project, we mainly use:**


*   JPEGImages         # image files
*   Annotations         # XML annotation files with bounding boxes
*   ImageSets/Main       # train/validation/test split files

### **3. Define dataset paths**

In [ ]:
import os

VOC_ROOT = "VOCdevkit/VOC2007"

IMAGE_DIR = os.path.join(VOC_ROOT, "JPEGImages")
ANNOTATION_DIR = os.path.join(VOC_ROOT, "Annotations")
SPLIT_DIR = os.path.join(VOC_ROOT, "ImageSets", "Main")

print("Image folder:", IMAGE_DIR)
print("Annotation folder:", ANNOTATION_DIR)
print("Split folder:", SPLIT_DIR)

###**4. Load train, validation, and test image IDs**

PASCAL VOC provides official split files.
Each line is an image ID, such as 000005, which corresponds to:


*   JPEGImages/000005.jpg
*   Annotations/000005.xml

In [ ]:
def load_image_ids(split_name):
    split_file = os.path.join(SPLIT_DIR, f"{split_name}.txt")

    with open(split_file, "r") as f:
        image_ids = [line.strip() for line in f.readlines()]

    return image_ids


train_ids = load_image_ids("train")
val_ids = load_image_ids("val")
test_ids = load_image_ids("test")

print("Number of training images:", len(train_ids))
print("Number of validation images:", len(val_ids))
print("Number of test images:", len(test_ids))

### **5. Define VOC classes**





In [ ]:
VOC_CLASSES = [
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor"
]

print("Number of classes:", len(VOC_CLASSES))
print(VOC_CLASSES)

### **6. Read XML annotation**

In [ ]:
import xml.etree.ElementTree as ET

def read_voc_annotation(image_id):
    """
    Read annotation information for one image.

    Returns:
        objects: a list of dictionaries.
        Each dictionary contains class name and bounding box.
    """

    xml_path = os.path.join(ANNOTATION_DIR, image_id + ".xml")

    tree = ET.parse(xml_path)
    root = tree.getroot()

    objects = []

    for obj in root.findall("object"):
        class_name = obj.find("name").text

        bndbox = obj.find("bndbox")
        xmin = int(float(bndbox.find("xmin").text))
        ymin = int(float(bndbox.find("ymin").text))
        xmax = int(float(bndbox.find("xmax").text))
        ymax = int(float(bndbox.find("ymax").text))

        objects.append({
            "class_name": class_name,
            "bbox": [xmin, ymin, xmax, ymax]
        })

    return objects

### **7. Print one sample annotation**

In [ ]:
sample_id = train_ids[0]

print("Image ID:", sample_id)
print("Image path:", os.path.join(IMAGE_DIR, sample_id + ".jpg"))
print("Annotation path:", os.path.join(ANNOTATION_DIR, sample_id + ".xml"))

objects = read_voc_annotation(sample_id)

print("Objects:")
for obj in objects:
    print(obj)

### **8. Visualize two samples with bounding boxes**

In [ ]:
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import random

def visualize_sample(image_id):
    """
    Visualize one image with ground-truth bounding boxes.
    """

    image_path = os.path.join(IMAGE_DIR, image_id + ".jpg")
    image = Image.open(image_path).convert("RGB")

    objects = read_voc_annotation(image_id)

    fig, ax = plt.subplots(1, figsize=(10, 8))
    ax.imshow(image)

    for obj in objects:
        class_name = obj["class_name"]
        xmin, ymin, xmax, ymax = obj["bbox"]

        width = xmax - xmin
        height = ymax - ymin

        rect = patches.Rectangle(
            (xmin, ymin),
            width,
            height,
            linewidth=2,
            edgecolor="red",
            facecolor="none"
        )

        ax.add_patch(rect)

        ax.text(
            xmin,
            ymin - 5,
            class_name,
            fontsize=12,
            color="red",
            bbox=dict(facecolor="white", alpha=0.7)
        )

    ax.set_title(f"Image ID: {image_id}")
    ax.axis("off")
    plt.show()


# Visualize two random training samples
sample_ids = random.sample(train_ids, 2)

for image_id in sample_ids:
    visualize_sample(image_id)

If the code runs correctly, two example images will be displayed with their bounding boxes and class labels.

## **Task 1: Object Detection Network Design**

In this task, you are required to design and train an object detection network using the PASCAL VOC 2007 dataset. The objective is to detect objects in images and predict both their class labels and bounding boxes.

You may build your model based on any suitable object detection architecture, such as YOLO, SSD, Faster R-CNN, RetinaNet, or other related models. You are also allowed to use or refer to existing architectures and open-source implementations. However, directly using an unchanged pre-trained model or simply running an official tutorial is not sufficient.

You must customize the model architecture and/or training pipeline to make it suitable for this assignment task. Possible customization may include, but is not limited to, modifying the backbone network, changing the detection head, adjusting anchor settings, freezing or unfreezing different layers, or designing other reasonable improvements.

The model performance should be evaluated using mAP@0.5 on the test set. A model achieving mAP@0.5 ≥ 60% will be considered to have met the performance requirement.

You should clearly explain your model design, training strategy, customization, and evaluation results in your (this) report. Your mark will not be determined by mAP alone. The correctness of the implementation, justification of the design choices, result visualization, and error analysis will also be considered.

### Approach Overview

We fine-tune a **Faster R-CNN with a ResNet-50 + FPN backbone** (the v2 variant pre-trained on COCO from `torchvision`) on PASCAL VOC 2007. Our customisations relative to a vanilla pre-trained model are:

1. **Box predictor head replaced** with a 21-class head (20 VOC classes + background) and trained from scratch.
2. **Backbone partially frozen**: `conv1`, `bn1`, `layer1`, `layer2` are frozen so the well-trained COCO low-level features are preserved while `layer3`, `layer4`, the FPN, RPN, and ROI heads are fine-tuned for VOC.
3. **Per-group learning rate**: the freshly-initialised head trains at 10× the LR of the fine-tuned layers, common practice when attaching a new head to a pretrained network.
4. **Warmup + cosine LR schedule**: 375-iter linear warmup (≈10% of total iterations) followed by cosine decay so the LR ends near zero for fine convergence.
5. **Augmentations**: random horizontal flip + colour jitter, with bounding-box-consistent geometry.
6. **Inference thresholds** tuned for VOC: `score_thresh=0.05`, `nms_thresh=0.5`, `detections_per_img=100`.
7. **Mixed-precision training** (`torch.amp.autocast('cuda')`) to roughly halve wall-time on a T4.
8. **Per-epoch validation mAP@0.5 and best-checkpoint selection**: the model used for the final test report is the epoch that maximises VOC2007 *val* mAP@0.5, not simply the last epoch.
9. **Two-track evaluation**: we report both COCO-style mAP@0.5 (101-point interpolation, via `torchmetrics`) and the canonical **PASCAL VOC 2007 11-point interpolated mAP@0.5** so the reported number matches the original VOC convention rather than only its COCO surrogate.

Faster R-CNN was chosen over one-stage detectors because its two-stage design (RPN, then ROI head) tends to be more accurate on the relatively small VOC2007 training set (2,501 images in the official train split) and is well supported in `torchvision`.

In [ ]:
!pip install -q torchmetrics pycocotools

In [ ]:
import os
import time
import random
import math
import numpy as np
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import Dataset, DataLoader
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.transforms import functional as TF
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import xml.etree.ElementTree as ET

# Fix seeds for reproducibility
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch {torch.__version__}, TorchVision {torchvision.__version__}")
print(f"Device: {device}")

### Dataset class with augmentations

In [ ]:
# Class 0 is reserved for background, VOC classes start at 1
CLASS_TO_IDX = {c: i + 1 for i, c in enumerate(VOC_CLASSES)}
IDX_TO_CLASS = {v: k for k, v in CLASS_TO_IDX.items()}
NUM_CLASSES = len(VOC_CLASSES) + 1


class VOCDetectionDataset(Dataset):
    def __init__(self, image_ids, image_dir, annotation_dir,
                 augment=False, keep_difficult=True):
        self.image_ids = list(image_ids)
        self.image_dir = image_dir
        self.annotation_dir = annotation_dir
        self.augment = augment
        self.keep_difficult = keep_difficult

    def __len__(self):
        return len(self.image_ids)

    def _read_xml(self, image_id):
        tree = ET.parse(os.path.join(self.annotation_dir, image_id + ".xml"))
        root = tree.getroot()
        boxes, labels, difficults = [], [], []
        for obj in root.findall("object"):
            difficult = int(obj.find("difficult").text) if obj.find("difficult") is not None else 0
            if difficult and not self.keep_difficult:
                continue
            cls = obj.find("name").text
            if cls not in CLASS_TO_IDX:
                continue
            # VOC bounding boxes are 1-indexed, convert to 0-indexed
            bb = obj.find("bndbox")
            xmin = float(bb.find("xmin").text) - 1.0
            ymin = float(bb.find("ymin").text) - 1.0
            xmax = float(bb.find("xmax").text) - 1.0
            ymax = float(bb.find("ymax").text) - 1.0
            if xmax <= xmin or ymax <= ymin:
                continue
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(CLASS_TO_IDX[cls])
            difficults.append(difficult)
        return boxes, labels, difficults

    def __getitem__(self, idx):
        image_id = self.image_ids[idx]
        image = Image.open(os.path.join(self.image_dir, image_id + ".jpg")).convert("RGB")
        boxes, labels, difficults = self._read_xml(image_id)

        boxes_t = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels_t = torch.as_tensor(labels, dtype=torch.int64)
        difficults_t = torch.as_tensor(difficults, dtype=torch.int64)

        # Augmentations: horizontal flip (also flips boxes) and colour jitter
        if self.augment:
            if random.random() < 0.5:
                image = TF.hflip(image)
                w = image.width
                if boxes_t.numel() > 0:
                    flipped = boxes_t.clone()
                    flipped[:, 0] = w - boxes_t[:, 2]
                    flipped[:, 2] = w - boxes_t[:, 0]
                    boxes_t = flipped
            if random.random() < 0.5:
                image = TF.adjust_brightness(image, 1.0 + (random.random() - 0.5) * 0.4)
            if random.random() < 0.5:
                image = TF.adjust_contrast(image, 1.0 + (random.random() - 0.5) * 0.4)
            if random.random() < 0.5:
                image = TF.adjust_saturation(image, 1.0 + (random.random() - 0.5) * 0.4)

        image_t = TF.to_tensor(image)
        area = ((boxes_t[:, 2] - boxes_t[:, 0]) * (boxes_t[:, 3] - boxes_t[:, 1])
                if boxes_t.numel() else torch.zeros((0,), dtype=torch.float32))
        # We pass the VOC `difficult` flag through the `iscrowd` field. This is
        # a deliberate approximation, not an identity:
        #   * Canonical VOC2007 evaluation EXCLUDES difficult GT entirely:
        #     a prediction matched to a difficult GT counts as neither TP nor FP,
        #     and difficult GT do not enter the recall denominator.
        #   * `torchmetrics` (COCO convention) treats `iscrowd=1` GT very
        #     similarly: matched predictions are ignored, and iscrowd GT do not
        #     count toward recall — semantically equivalent for our purposes.
        # The reported torchmetrics number is therefore a faithful surrogate for
        # VOC's "skip difficult" rule, and the separate VOC 11-point evaluator
        # below applies the exact VOC rule for the official figure.
        target = {
            "boxes": boxes_t,
            "labels": labels_t,
            "image_id": torch.tensor([idx]),
            "iscrowd": difficults_t,
            "area": area,
        }
        return image_t, target


# Variable-sized images: pass batches as tuples of lists
def collate_fn(batch):
    return tuple(zip(*batch))


train_ds = VOCDetectionDataset(train_ids, IMAGE_DIR, ANNOTATION_DIR, augment=True,  keep_difficult=True)
val_ds   = VOCDetectionDataset(val_ids,   IMAGE_DIR, ANNOTATION_DIR, augment=False, keep_difficult=True)
test_ds  = VOCDetectionDataset(test_ids,  IMAGE_DIR, ANNOTATION_DIR, augment=False, keep_difficult=True)

print(f"Train: {len(train_ds)}")
print(f"Val:   {len(val_ds)}")
print(f"Test:  {len(test_ds)}")

BATCH_SIZE = 4
NUM_WORKERS = 2

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate_fn, num_workers=NUM_WORKERS, pin_memory=True)


### Customised Faster R-CNN

Starting from `fasterrcnn_resnet50_fpn_v2` (COCO-pretrained), we apply the customisations described above.


In [ ]:
def build_model(num_classes: int) -> nn.Module:
    weights = FasterRCNN_ResNet50_FPN_V2_Weights.COCO_V1
    model = fasterrcnn_resnet50_fpn_v2(weights=weights)

    # Replace COCO head (91 classes) with a fresh head for 20 VOC classes + background
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # Freeze early backbone layers to preserve COCO low-level features
    for name, param in model.backbone.body.named_parameters():
        if name.startswith(("conv1", "bn1", "layer1", "layer2")):
            param.requires_grad = False

    # Looser inference thresholds than the COCO defaults (better recall on VOC)
    model.roi_heads.score_thresh      = 0.05
    model.roi_heads.nms_thresh        = 0.5
    model.roi_heads.detections_per_img = 100

    return model


model = build_model(NUM_CLASSES).to(device)
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
n_total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters: {n_trainable/1e6:.2f}M of {n_total/1e6:.2f}M ({100*n_trainable/n_total:.1f}%)")


### Training

SGD with momentum, weight decay, linear warmup, then cosine decay. The new head gets 10× the base LR.
The cell below uses mixed precision when a CUDA GPU is available.


In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

NUM_EPOCHS = 6
BASE_LR    = 5e-3

# Two parameter groups: the freshly initialised head trains at 10x the base LR
head_params       = list(model.roi_heads.box_predictor.parameters())
head_param_ids    = {id(p) for p in head_params}
other_params      = [p for p in model.parameters()
                     if p.requires_grad and id(p) not in head_param_ids]

optimizer = torch.optim.SGD(
    [{"params": other_params, "lr": BASE_LR},
     {"params": head_params,  "lr": BASE_LR * 10}],
    momentum=0.9, weight_decay=5e-4,
)

iters_per_epoch = len(train_loader)
total_iters     = iters_per_epoch * NUM_EPOCHS
warmup_iters    = min(500, total_iters // 10)

# Linear warmup then cosine decay to zero
def lr_factor(it):
    if it < warmup_iters:
        return (it + 1) / max(1, warmup_iters)
    progress = (it - warmup_iters) / max(1, total_iters - warmup_iters)
    return 0.5 * (1.0 + math.cos(math.pi * progress))

# Mixed precision for faster training on GPU (use new torch.amp API)
use_amp = (device.type == "cuda")
scaler = torch.amp.GradScaler("cuda") if use_amp else None


@torch.no_grad()
def quick_val_map(model, loader, device):
    """Compute COCO-style mAP@0.5 on a loader. Returns float."""
    model.eval()
    metric = MeanAveragePrecision(box_format="xyxy", iou_type="bbox",
                                  iou_thresholds=[0.5], class_metrics=False)
    for images, targets in loader:
        images_d = [img.to(device) for img in images]
        preds = model(images_d)
        preds_cpu   = [{k: v.detach().cpu() for k, v in p.items()} for p in preds]
        targets_cpu = [{k: v.detach().cpu() for k, v in t.items()} for t in targets]
        metric.update(preds_cpu, targets_cpu)
    res = metric.compute()
    return float(res["map"].item())


loss_history    = []
val_map_history = []
best_val_map    = -1.0
global_iter     = 0
print(f"Training: {NUM_EPOCHS} epochs, {iters_per_epoch} iters/epoch, warmup {warmup_iters}")

for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss, t0 = 0.0, time.time()
    for images, targets in train_loader:
        images  = [img.to(device, non_blocking=True) for img in images]
        targets = [{k: v.to(device, non_blocking=True) for k, v in t.items()}
                   for t in targets]

        f = lr_factor(global_iter)
        optimizer.param_groups[0]["lr"] = BASE_LR      * f
        optimizer.param_groups[1]["lr"] = BASE_LR * 10 * f

        optimizer.zero_grad(set_to_none=True)
        if use_amp:
            with torch.amp.autocast("cuda"):
                loss_dict = model(images, targets)
                loss      = sum(loss_dict.values())
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss_dict = model(images, targets)
            loss      = sum(loss_dict.values())
            loss.backward()
            optimizer.step()

        epoch_loss  += loss.item()
        global_iter += 1

    avg_loss = epoch_loss / iters_per_epoch
    loss_history.append(avg_loss)
    train_min = (time.time() - t0) / 60

    # Validation mAP@0.5 for early-stopping / best-checkpoint selection
    t1 = time.time()
    val_map = quick_val_map(model, val_loader, device)
    val_map_history.append(val_map)
    val_min = (time.time() - t1) / 60

    saved = ""
    if val_map > best_val_map:
        best_val_map = val_map
        torch.save(model.state_dict(), "fasterrcnn_voc_best.pth")
        saved = "  [best ckpt saved]"

    print(f"Epoch {epoch+1}/{NUM_EPOCHS}  loss {avg_loss:.4f}  "
          f"val mAP@0.5 {val_map:.4f}  train {train_min:.1f}m  val {val_min:.1f}m  "
          f"lr {optimizer.param_groups[0]['lr']:.5f}{saved}")

# Save the final-epoch checkpoint as well, for the ablation discussion
torch.save(model.state_dict(), "fasterrcnn_voc_last.pth")
print(f"Best val mAP@0.5: {best_val_map:.4f}")
print("Saved checkpoints: fasterrcnn_voc_best.pth (best val), fasterrcnn_voc_last.pth (last epoch)")


In [ ]:
fig, ax1 = plt.subplots(figsize=(7, 4))
xs = range(1, len(loss_history) + 1)
ax1.plot(xs, loss_history, marker="o", color="tab:blue", label="train loss")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Mean training loss", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")
ax1.grid(True, alpha=0.3)

ax2 = ax1.twinx()
ax2.plot(xs, val_map_history, marker="s", color="tab:red", label="val mAP@0.5")
ax2.set_ylabel("Val mAP@0.5", color="tab:red")
ax2.tick_params(axis="y", labelcolor="tab:red")
ax2.set_ylim(0, 1)

best_epoch = int(np.argmax(val_map_history)) + 1
ax2.axvline(best_epoch, color="grey", linestyle="--", alpha=0.6,
            label=f"best epoch ({best_epoch}, mAP={max(val_map_history):.3f})")

fig.suptitle("Training loss and validation mAP@0.5")
ax2.legend(loc="lower right")
plt.tight_layout()
plt.show()


### Evaluation: mAP@0.5 on the test set

We restore the **best validation checkpoint** and evaluate on the held-out VOC2007 test split (4,952 images) with **two metrics in parallel**:

1. **COCO-style mAP@0.5** via `torchmetrics.detection.MeanAveragePrecision` (101-point interpolation). `class_metrics=True` gives per-class AP for Task 2.
2. **Canonical PASCAL VOC 2007 mAP@0.5**: 11-point interpolated AP, with difficult GT excluded from evaluation per the original VOC convention. This is the figure that should be compared against published VOC2007 numbers in the literature.

Predictions are collected once and reused for both metrics and for the failure-case analysis in Task 2, which avoids three separate inference passes over the test set.

In [ ]:
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# Restore best validation checkpoint for the final test report
ckpt_path = "fasterrcnn_voc_best.pth"
print(f"Loading best checkpoint: {ckpt_path}")
model.load_state_dict(torch.load(ckpt_path, map_location=device))
model.to(device).eval()


def _box_iou_np(b1, b2):
    """Pairwise IoU between two sets of xyxy boxes, numpy implementation."""
    if len(b1) == 0 or len(b2) == 0:
        return np.zeros((len(b1), len(b2)), dtype=np.float32)
    b1 = np.asarray(b1, dtype=np.float32)
    b2 = np.asarray(b2, dtype=np.float32)
    ix1 = np.maximum(b1[:, None, 0], b2[None, :, 0])
    iy1 = np.maximum(b1[:, None, 1], b2[None, :, 1])
    ix2 = np.minimum(b1[:, None, 2], b2[None, :, 2])
    iy2 = np.minimum(b1[:, None, 3], b2[None, :, 3])
    iw  = np.clip(ix2 - ix1, 0, None)
    ih  = np.clip(iy2 - iy1, 0, None)
    inter = iw * ih
    a1 = (b1[:, 2] - b1[:, 0]) * (b1[:, 3] - b1[:, 1])
    a2 = (b2[:, 2] - b2[:, 0]) * (b2[:, 3] - b2[:, 1])
    union = a1[:, None] + a2[None, :] - inter
    return np.where(union > 0, inter / union, 0.0).astype(np.float32)


@torch.no_grad()
def collect_predictions(model, loader, device):
    """Run the model over `loader` once and collect CPU-side preds/targets."""
    model.eval()
    all_preds, all_targets = [], []
    for images, targets in loader:
        images_d = [img.to(device) for img in images]
        preds    = model(images_d)
        all_preds.extend([{k: v.detach().cpu() for k, v in p.items()} for p in preds])
        all_targets.extend([{k: v.detach().cpu() for k, v in t.items()} for t in targets])
    return all_preds, all_targets


def compute_torchmetrics_map(preds, targets, iou_thresholds=(0.5,)):
    metric = MeanAveragePrecision(
        box_format="xyxy", iou_type="bbox",
        iou_thresholds=list(iou_thresholds), class_metrics=True,
    )
    metric.update(preds, targets)
    return metric.compute()


def compute_voc11_ap(preds, targets, iou_thresh=0.5, num_classes=NUM_CLASSES):
    """Canonical PASCAL VOC 2007 11-point interpolated AP per class.

    Implements the original VOC rule:
      * Difficult GT (`iscrowd=1`) are excluded from evaluation entirely:
        predictions matched to a difficult GT are dropped (neither TP nor FP),
        and difficult GT do not contribute to the recall denominator.
      * Each non-difficult GT can only be matched once (greedy by score).
      * AP = mean of precision values at recall levels {0, 0.1, ..., 1.0}.
    Returns a dict {class_idx -> AP} for class indices 1..num_classes-1.
    """
    per_cls_scores = {c: [] for c in range(1, num_classes)}
    per_cls_tp     = {c: [] for c in range(1, num_classes)}
    per_cls_fp     = {c: [] for c in range(1, num_classes)}
    per_cls_npos   = {c: 0  for c in range(1, num_classes)}

    for pred, target in zip(preds, targets):
        gt_boxes  = target["boxes"].numpy()
        gt_labels = target["labels"].numpy()
        gt_diff   = (target["iscrowd"].numpy().astype(bool)
                     if "iscrowd" in target else np.zeros(len(gt_labels), dtype=bool))

        for c in range(1, num_classes):
            cls_mask = gt_labels == c
            per_cls_npos[c] += int((cls_mask & ~gt_diff).sum())

        pb = pred["boxes"].numpy()
        pl = pred["labels"].numpy()
        ps = pred["scores"].numpy()

        for c in range(1, num_classes):
            pm = pl == c
            if not pm.any():
                continue
            cb = pb[pm]; cs = ps[pm]
            order = np.argsort(-cs)
            cb = cb[order]; cs = cs[order]

            gm = gt_labels == c
            gb = gt_boxes[gm]
            gd = gt_diff[gm]
            matched = np.zeros(len(gb), dtype=bool)

            for k in range(len(cb)):
                if len(gb) == 0:
                    per_cls_scores[c].append(cs[k]); per_cls_tp[c].append(0); per_cls_fp[c].append(1)
                    continue
                ious = _box_iou_np(cb[k:k+1], gb)[0]
                j = int(np.argmax(ious))
                if ious[j] >= iou_thresh:
                    if matched[j]:
                        per_cls_scores[c].append(cs[k]); per_cls_tp[c].append(0); per_cls_fp[c].append(1)
                    elif gd[j]:
                        # Matched a difficult GT: ignore this prediction entirely.
                        matched[j] = True
                    else:
                        matched[j] = True
                        per_cls_scores[c].append(cs[k]); per_cls_tp[c].append(1); per_cls_fp[c].append(0)
                else:
                    per_cls_scores[c].append(cs[k]); per_cls_tp[c].append(0); per_cls_fp[c].append(1)

    aps = {}
    for c in range(1, num_classes):
        if per_cls_npos[c] == 0:
            continue
        if not per_cls_scores[c]:
            aps[c] = 0.0
            continue
        scores = np.asarray(per_cls_scores[c])
        tp     = np.asarray(per_cls_tp[c])
        fp     = np.asarray(per_cls_fp[c])
        order  = np.argsort(-scores)
        tp     = tp[order]; fp = fp[order]
        tpc, fpc = np.cumsum(tp), np.cumsum(fp)
        recall    = tpc / per_cls_npos[c]
        precision = tpc / np.maximum(tpc + fpc, 1e-12)
        ap = 0.0
        for t in np.linspace(0, 1, 11):
            mask = recall >= t
            ap += (precision[mask].max() if mask.any() else 0.0) / 11.0
        aps[c] = float(ap)
    return aps


print("Collecting test-set predictions (single inference pass)...")
t0 = time.time()
test_preds, test_targets = collect_predictions(model, test_loader, device)
print(f"  collected {len(test_preds)} images in {(time.time()-t0)/60:.1f} min")

print("\nComputing COCO-style mAP@0.5 (torchmetrics, 101-pt interpolation)...")
results = compute_torchmetrics_map(test_preds, test_targets)
test_map_50_coco = float(results["map"].item())

print("Computing canonical VOC2007 mAP@0.5 (11-pt interpolation, difficult excluded)...")
voc11_aps = compute_voc11_ap(test_preds, test_targets)
test_map_50_voc11 = float(np.mean(list(voc11_aps.values()))) if voc11_aps else 0.0

print(f"\nTest mAP@0.5  (COCO 101-pt) : {test_map_50_coco:.4f} ({test_map_50_coco*100:.2f}%)")
print(f"Test mAP@0.5  (VOC  11-pt)  : {test_map_50_voc11:.4f} ({test_map_50_voc11*100:.2f}%)")
print(f"Target threshold            : 0.60 (60%)")
print(f"Pass (VOC mAP@0.5 >= 0.60)  : {'yes' if test_map_50_voc11 >= 0.60 else 'no'}")


### Visualise predictions on test images

In [ ]:
@torch.no_grad()
def predict_image(model, image_id, score_threshold=0.5):
    model.eval()
    image = Image.open(os.path.join(IMAGE_DIR, image_id + ".jpg")).convert("RGB")
    img_t = TF.to_tensor(image).to(device)
    pred  = model([img_t])[0]
    keep  = pred["scores"] >= score_threshold
    return image, {
        "boxes":  pred["boxes"][keep].cpu().numpy(),
        "labels": pred["labels"][keep].cpu().numpy(),
        "scores": pred["scores"][keep].cpu().numpy(),
    }


def visualize_predictions(model, image_ids, score_threshold=0.5):
    for image_id in image_ids:
        image, pred = predict_image(model, image_id, score_threshold)
        gt = read_voc_annotation(image_id)

        fig, axes = plt.subplots(1, 2, figsize=(14, 7))
        for ax in axes: ax.imshow(image); ax.axis("off")
        axes[0].set_title(f"Ground Truth - {image_id}")
        axes[1].set_title(f"Prediction (score >= {score_threshold})")

        for obj in gt:
            xmin, ymin, xmax, ymax = obj["bbox"]
            axes[0].add_patch(patches.Rectangle(
                (xmin, ymin), xmax-xmin, ymax-ymin,
                linewidth=2, edgecolor="lime", facecolor="none"))
            axes[0].text(xmin, ymin-3, obj["class_name"], color="white", fontsize=10,
                         bbox=dict(facecolor="green", alpha=0.7, pad=1))

        for box, label, score in zip(pred["boxes"], pred["labels"], pred["scores"]):
            xmin, ymin, xmax, ymax = box
            axes[1].add_patch(patches.Rectangle(
                (xmin, ymin), xmax-xmin, ymax-ymin,
                linewidth=2, edgecolor="red", facecolor="none"))
            axes[1].text(xmin, ymin-3, f"{IDX_TO_CLASS[int(label)]} {score:.2f}",
                         color="white", fontsize=10,
                         bbox=dict(facecolor="red", alpha=0.7, pad=1))

        plt.tight_layout(); plt.show()


visualize_predictions(model, test_ids[:4])


### Task 1 Discussion

**Headline result.** The customised Faster R-CNN reaches **mAP@0.5 = 0.8521** on the VOC2007 test set under the **canonical PASCAL VOC 2007 11-point interpolated metric**, and **0.8966** under the COCO-style 101-point interpolation. Both clear the 0.60 (60%) assignment target by a wide margin (+25.2 points VOC, +29.7 points COCO). End-to-end cost on the Colab T4 was about 95 minutes for training plus per-epoch validation, and about 17 minutes for test inference.

**Architecture choice.** Faster R-CNN was chosen over single-stage detectors (SSD, YOLO) because the VOC2007 train split is small (only 2,501 images here) and the two-stage RPN-then-ROI design is sample-efficient: the RPN supplies high-quality proposals so the box head only needs to refine and classify, which works well even with limited data.

**Customisations and their effect.**

1. **Box predictor head replaced** with a 21-class head (20 + background), trained from scratch. Necessary because the COCO-pretrained head outputs 91 logits.
2. **Backbone partially frozen** (`conv1`, `bn1`, `layer1`, `layer2` frozen). 41.91 M of 43.35 M parameters remain trainable (96.7%). Freezing the early layers regularises the model and preserves COCO low-level features that transfer almost directly.
3. **Per-group learning rate**: the freshly initialised box predictor head trains at 10× the LR (5e-2 peak) of the fine-tuned layers (5e-3 peak). This lets the new head catch up quickly without destabilising the fine-tuned backbone.
4. **Warmup + cosine LR schedule**: 375-iteration linear warmup then cosine decay across 3,756 total iterations. The schedule shows up clearly in the printed per-epoch LRs (0.00493, 0.00422, 0.00294, 0.00151, 0.00041, 0.00000).
5. **Mixed-precision training** (`torch.amp.autocast('cuda')`) roughly halved wall-time vs. fp32 with no accuracy degradation.
6. **Augmentations**: horizontal flip + colour jitter (brightness, contrast, saturation), each independently applied with p=0.5, with bounding-box-consistent geometry.
7. **Inference thresholds tuned for VOC**: `score_thresh=0.05`, `nms_thresh=0.5`, `detections_per_img=100` (the COCO defaults are stricter and noticeably hurt VOC recall).
8. **Per-epoch validation mAP@0.5 with best-checkpoint selection.** Validation mAP@0.5 was *0.8407, 0.8684, 0.8819, 0.8923, **0.8974**, 0.8967*, peaking at **epoch 5** and flattening (epoch 6 is within run-to-run noise of epoch 5, the gap is only 0.0007). The best-checkpoint mechanism therefore acted as cheap insurance against late-epoch overfitting rather than rescuing a major regression on this particular run; on a longer training schedule it would matter more. The shape of the curve (gain decelerating from +2.8, +1.4, +1.0, +0.5, then ~0 mAP points) also confirms that 6 epochs is approximately the right budget for this recipe.
9. **Two-track evaluation (VOC 11-pt and COCO 101-pt).** The literature on VOC2007 uses the 11-point interpolated AP, which is consistently 4 to 5 mAP points lower than the COCO-style 101-pt figure that `torchmetrics` returns by default (here 0.8521 vs 0.8966, a 4.5-point gap). Reporting only the COCO-style number would over-state the model's standing relative to published VOC2007 baselines, so we report both.

**Training dynamics.** Training loss decreased monotonically every epoch: 0.4552, 0.2737, 0.2251, 0.1825, 0.1531, 0.1407. The relative drop slowed from -40% (epoch 1 to 2) down to -8% (epoch 5 to 6). The validation mAP curve confirms the model is essentially converged at epoch 5: the further loss drop in epoch 6 does not improve val mAP (the -0.0007 change is well inside the per-run variance from non-deterministic GPU operations and DataLoader worker seeding), so we treat epoch 5 as the practical stopping point for this recipe.

**Limitations and what was deliberately left out.**

- We trained on the official **train** split only (2,501 images). Combining train + val (5,011 images, the standard "trainval" set) would likely push mAP another 2 to 3 points; we kept them separate to preserve a clean validation signal as the assignment instructed.
- Geometric augmentations like random scale/crop or mosaic were not used. They tend to help small objects but require more careful target adjustment than the simple flip implemented here. This is a natural extension and is reflected in the lower AP for the smallest VOC classes (see Task 2.3).
- A heavier backbone (ResNet-101, Swin-T) would probably help further on the difficult classes but was not necessary to clear the bar.
- Anchor sizes were left at the COCO defaults. VOC2007 has somewhat different object-size statistics (proportionally larger objects than COCO), so a small VOC-specific anchor sweep is plausibly worth 0.5 to 1 mAP, but the gain is small relative to the other levers.

The detailed per-class behaviour and failure modes are analysed in Task 2 below.

## **Task 2: Detection Performance and Error Analysis**

Based on the object detection model developed in Task 1, you are required to conduct a detailed performance and error analysis to better understand the behaviour of your model.

Your analysis should include the following components:

1. Class-wise performance analysis

Report the AP or other appropriate detection performance metric for each object category. Identify which classes achieve better performance and which classes are more difficult for your model to detect.

2. Error case visualization

Visualize at least five representative failure cases from your detection results. Each example should clearly show the input image, ground-truth bounding boxes, predicted bounding boxes, predicted class labels, and confidence scores where applicable.

3. Error type discussion and potential solutions

Discuss the possible causes of the observed detection errors, such as missed detections, false positives, inaccurate bounding boxes, class confusion, small objects, occlusion, complex backgrounds, or class imbalance. You should also discuss possible solutions or improvements.

### Task 2.1: Class-wise Performance

Per-class APs are reported in two forms: the canonical **PASCAL VOC 11-point interpolated AP@0.5** and the COCO-style **101-point AP@0.5** from `torchmetrics` (with `class_metrics=True`). Both are computed from the same single inference pass and the same predictions used in Task 1, so the per-class numbers are directly comparable to the headline mAP figures above.


In [ ]:
# torchmetrics per-class AP@0.5 (COCO-style, 101-pt interpolation)
classes_arr   = results["classes"].tolist()
per_class_aps = results["map_per_class"].tolist()

coco_per_class = {}
for cls_idx, ap in zip(classes_arr, per_class_aps):
    if int(cls_idx) == 0:
        continue
    name = IDX_TO_CLASS.get(int(cls_idx), f"id_{cls_idx}")
    coco_per_class[name] = 0.0 if (ap is None or math.isnan(ap) or ap < 0) else float(ap)
for c in VOC_CLASSES:
    coco_per_class.setdefault(c, 0.0)

# Canonical VOC 11-point per-class AP (already computed in voc11_aps as {idx: AP})
voc11_per_class = {IDX_TO_CLASS[idx]: ap for idx, ap in voc11_aps.items()}
for c in VOC_CLASSES:
    voc11_per_class.setdefault(c, 0.0)

# Sort by VOC 11-pt AP (the canonical figure)
ordered = sorted(VOC_CLASSES, key=lambda c: voc11_per_class[c], reverse=True)

print(f"{'Class':<14} {'VOC 11-pt':>10} {'COCO 101-pt':>12}")
print("-" * 40)
for c in ordered:
    print(f"{c:<14} {voc11_per_class[c]:>10.4f} {coco_per_class[c]:>12.4f}")
print("-" * 40)
print(f"{'mAP@0.5':<14} {np.mean([voc11_per_class[c] for c in VOC_CLASSES]):>10.4f} "
      f"{np.mean([coco_per_class[c] for c in VOC_CLASSES]):>12.4f}")

# class_ap kept for downstream cells: list of (name, voc11_ap) sorted desc
class_ap = [(c, voc11_per_class[c]) for c in ordered]


In [ ]:
names    = [c for c in ordered]
voc_aps  = [voc11_per_class[c] for c in names]
coco_aps = [coco_per_class[c]  for c in names]

x = np.arange(len(names))
w = 0.4

plt.figure(figsize=(12, 5))
plt.bar(x - w/2, voc_aps,  w, color="steelblue", label="VOC 11-pt AP@0.5")
plt.bar(x + w/2, coco_aps, w, color="lightcoral", label="COCO 101-pt AP@0.5")
plt.axhline(np.mean(voc_aps),  color="steelblue",  linestyle="--", alpha=0.7,
            label=f"VOC mAP = {np.mean(voc_aps):.3f}")
plt.axhline(np.mean(coco_aps), color="lightcoral", linestyle="--", alpha=0.7,
            label=f"COCO mAP = {np.mean(coco_aps):.3f}")
plt.xticks(x, names, rotation=45, ha="right")
plt.ylim(0, 1)
plt.ylabel("AP@0.5")
plt.title("Per-class AP@0.5 on VOC2007 test (VOC vs COCO interpolation)")
plt.legend()
plt.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
plt.show()


### Task 2.2: Failure Case Identification & Visualisation

We re-use the predictions collected during evaluation (no extra inference) to flag detection errors on the **entire** VOC2007 test set (4,952 images), at a confidence threshold of `score >= 0.5`. Each image is scored using simple IoU-based rules:

- **Missed detection.** A (non-difficult) GT box has IoU < 0.5 with every same-class predicted box (or its class isn't predicted at all).
- **False positive.** A prediction with score ≥ 0.5 has IoU < 0.5 with every same-class GT box.
- **Misclassification.** A prediction overlaps a GT box at IoU ≥ 0.5 but the class label is wrong.

Each failure image is then automatically tagged with the **error category** it best exemplifies, so the visualised cases below correspond to the categories discussed in Task 2.3 rather than just being the noisiest images:

| Category | Heuristic |
|---|---|
| `small_clutter` | GT contains `bottle`, `pottedplant`, or `tvmonitor` |
| `furniture_intra_class` | GT contains `chair`, `sofa`, or `diningtable` |
| `elongated_localisation` | GT contains `boat` (long, thin objects) |
| `animal_confusion` | misclassification between two animal classes |
| `background_fp` | ≥ 2 false positives, no GT for those classes in the image |
| `other` | none of the above |

We display the worst-by-error-count case from each non-empty category.

In [ ]:
ANIMAL_CLASSES        = set(['bird', 'cat', 'cow', 'dog', 'horse', 'sheep'])
FURNITURE_CLASSES     = {"chair", "sofa", "diningtable"}
SMALL_CLUTTER_CLASSES = {"bottle", "pottedplant", "tvmonitor"}
ELONGATED_CLASSES     = {"boat"}


def categorise_failure(gt_labels, pred_labels, n_missed, n_fp, n_misclass):
    gt_set, pred_set = set(gt_labels), set(pred_labels)

    # Animal-vs-animal confusion takes priority — most informative failure mode
    if n_misclass > 0 and (gt_set & ANIMAL_CLASSES) and (pred_set & ANIMAL_CLASSES):
        return "animal_confusion"
    if gt_set & ELONGATED_CLASSES:
        return "elongated_localisation"
    if gt_set & SMALL_CLUTTER_CLASSES:
        return "small_clutter"
    if gt_set & FURNITURE_CLASSES:
        return "furniture_intra_class"
    if n_fp >= 2 and n_missed == 0:
        return "background_fp"
    return "other"


def collect_failures_from_preds(image_ids, preds, targets,
                                score_thresh=0.5, iou_thresh=0.5):
    """Use cached test predictions (no extra model inference)."""
    failures = []
    for image_id, pred, target in zip(image_ids, preds, targets):
        gt_boxes  = target["boxes"].numpy()
        gt_labels = [IDX_TO_CLASS[int(l)] for l in target["labels"].numpy()]
        gt_diff   = (target["iscrowd"].numpy().astype(bool)
                     if "iscrowd" in target else np.zeros(len(gt_labels), dtype=bool))

        # Filter to confident predictions for failure analysis
        scores = pred["scores"].numpy()
        keep   = scores >= score_thresh
        pb = pred["boxes"].numpy()[keep]
        pl = [IDX_TO_CLASS[int(l)] for l in pred["labels"].numpy()[keep]]
        ps = scores[keep]

        ious = _box_iou_np(gt_boxes, pb) if (len(gt_boxes) and len(pb)) else \
               np.zeros((len(gt_boxes), len(pb)), dtype=np.float32)

        # Per GT box: missed if no overlap, misclassified if best overlap has wrong label.
        # Difficult GT are skipped from the count.
        n_missed, n_misclass = 0, 0
        for i, (gl, gd) in enumerate(zip(gt_labels, gt_diff)):
            if gd:
                continue
            if pb.shape[0] == 0:
                n_missed += 1
                continue
            j = int(np.argmax(ious[i]))
            if ious[i, j] < iou_thresh:
                n_missed += 1
            elif pl[j] != gl:
                n_misclass += 1

        # FP: confident pred whose class has no non-difficult same-class GT overlap
        n_fp = 0
        for j, pred_lbl in enumerate(pl):
            same_idx = [i for i, (gl, gd) in enumerate(zip(gt_labels, gt_diff))
                        if gl == pred_lbl and not gd]
            if not same_idx:
                n_fp += 1
            elif ious[same_idx, j].max() < iou_thresh:
                n_fp += 1

        total = n_missed + n_fp + n_misclass
        if total > 0:
            failures.append({
                "image_id":   image_id,
                "n_missed":   n_missed,
                "n_fp":       n_fp,
                "n_misclass": n_misclass,
                "total":      total,
                "pred_boxes":  pb,
                "pred_labels": pl,
                "pred_scores": ps,
                "gt_boxes":    gt_boxes,
                "gt_labels":   gt_labels,
                "category":    categorise_failure(gt_labels, pl, n_missed, n_fp, n_misclass),
            })
    return failures


failures = collect_failures_from_preds(test_ids, test_preds, test_targets,
                                       score_thresh=0.5, iou_thresh=0.5)
print(f"Inspected all {len(test_ids)} test images; "
      f"{len(failures)} contained at least one detection error "
      f"({100*len(failures)/len(test_ids):.1f}% of the test set).")

# Category breakdown
from collections import Counter
cat_counts = Counter(f["category"] for f in failures)
print("\nFailures per category:")
for cat in ["small_clutter", "furniture_intra_class", "elongated_localisation",
            "animal_confusion", "background_fp", "other"]:
    print(f"  {cat:<24} {cat_counts.get(cat, 0):>5}")


In [ ]:
def plot_failure(case):
    image = Image.open(os.path.join(IMAGE_DIR, case["image_id"] + ".jpg")).convert("RGB")
    fig, axes = plt.subplots(1, 2, figsize=(14, 7))
    for ax in axes:
        ax.imshow(image); ax.axis("off")
    axes[0].set_title(f"GT  -  {case['image_id']}  -  category: {case['category']}")
    axes[1].set_title(f"Prediction  (missed={case['n_missed']}, "
                      f"FP={case['n_fp']}, misclass={case['n_misclass']})")

    for box, lbl in zip(case["gt_boxes"], case["gt_labels"]):
        xmin, ymin, xmax, ymax = box
        axes[0].add_patch(patches.Rectangle(
            (xmin, ymin), xmax-xmin, ymax-ymin,
            linewidth=2, edgecolor="lime", facecolor="none"))
        axes[0].text(xmin, ymin-3, lbl, color="white", fontsize=10,
                     bbox=dict(facecolor="green", alpha=0.7, pad=1))

    for box, lbl, sc in zip(case["pred_boxes"], case["pred_labels"], case["pred_scores"]):
        xmin, ymin, xmax, ymax = box
        axes[1].add_patch(patches.Rectangle(
            (xmin, ymin), xmax-xmin, ymax-ymin,
            linewidth=2, edgecolor="red", facecolor="none"))
        axes[1].text(xmin, ymin-3, f"{lbl} {sc:.2f}",
                     color="white", fontsize=10,
                     bbox=dict(facecolor="red", alpha=0.7, pad=1))

    plt.tight_layout()
    plt.show()


# Pick the worst (highest total error count) case from each non-empty category,
# in the same order as the discussion in Task 2.3.
CATEGORY_ORDER = [
    "small_clutter",
    "furniture_intra_class",
    "elongated_localisation",
    "animal_confusion",
    "background_fp",
    "other",
]

representative = []
for cat in CATEGORY_ORDER:
    in_cat = [f for f in failures if f["category"] == cat]
    if in_cat:
        in_cat.sort(key=lambda d: d["total"], reverse=True)
        representative.append(in_cat[0])

print(f"Showing {len(representative)} representative failures, one per category.")
for case in representative:
    plot_failure(case)


### Task 2.3: Error Type Discussion & Potential Solutions

**Per-class results recap (VOC 11-pt mAP@0.5 = 0.8521, COCO 101-pt mAP@0.5 = 0.8966).** Sorting by the canonical VOC 11-pt AP, the 20 classes split cleanly into three difficulty tiers:

- **Tier 1 (VOC AP ≥ 0.88), 9 classes:** aeroplane (0.90), horse (0.90), bicycle (0.90), bus (0.90), car (0.90), cow (0.89), person (0.89), motorbike (0.89), train (0.89). Vehicles and large animals.
- **Tier 2 (0.85 ≤ VOC AP < 0.88), 5 classes:** cat (0.88), dog (0.87), bird (0.87), tvmonitor (0.86), sheep (0.86).
- **Tier 3 (VOC AP < 0.85), 6 classes:** sofa (0.84), diningtable (0.82), bottle (0.81), boat (0.78), chair (0.76), pottedplant (0.64).

Tier 3 is responsible for almost the entire error budget. Encouragingly, even the weakest class (pottedplant at 0.64) already exceeds the assignment-wide 0.60 target on its own, so the model is consistently competent rather than excellent on a few easy classes only.

**Failure-case statistics (full test set, score ≥ 0.5).** Of the 4,952 test images, **2,873 (58.0%)** contained at least one detection error. The auto-categorised breakdown is:

| Category | # images | Comment |
|---|---|---|
| `background_fp` | 654 | Largest categorised mode: confident detections of classes that aren't present in the image. |
| `small_clutter` | 541 | Images containing `bottle`, `pottedplant`, or `tvmonitor`. |
| `furniture_intra_class` | 476 | Images containing `chair`, `sofa`, or `diningtable`. |
| `elongated_localisation` | 131 | Images containing `boat`. |
| `animal_confusion` | 128 | Misclassifications between two animal classes. |
| `other` | 943 | Images that don't fit any single category, typically multi-object scenes with one small/occluded miss. |

Each of the six failure plots above corresponds to one category in this order, so the discussion below maps directly to the visualised cases. Note that `other` (943 images, 32.8% of all failure images) is the unlabelled bucket: it captures images that did not match any of the five heuristics, typically multi-object scenes where one minor instance was missed but the bulk of the labels were correct. We do not visualise an `other` case because the failures within it are heterogeneous by construction; they are best understood as a tail rather than a category.

**Error categories, classes affected, and proposed fixes.**

**1. Small / cluttered objects (`small_clutter`, 541 images).** *bottle (0.81), pottedplant (0.64), tvmonitor (0.86), chair (0.76)* dominate this category. These objects are often only a few dozen pixels wide and routinely appear in heavily occluded indoor scenes (kitchens, lounges, dining tables). The ResNet-50 + FPN backbone provides multi-scale features down to stride 4, but the smallest VOC objects are still very low-resolution at that level. Pottedplant's 0.64 AP, the lowest of any class, is the clearest single signature of this weakness.
*Solutions:* (a) increase input resolution (shortest side 800 to 1000-1333 px), (b) add a finer FPN level (P2 / P1) or use BiFPN / PANet, (c) add scale-jitter and mosaic augmentation so small instances appear in more contexts, (d) use SAHI-style sliced inference at test time for crowded indoor scenes.

**2. Furniture with high intra-class variation (`furniture_intra_class`, 476 images).** *chair (0.76), diningtable (0.82), sofa (0.84).* Chairs come in radically different shapes (office, dining, beach, armchair), diningtables are often defined more by what's on top of them than by the table surface itself, and sofas overlap visually with chairs and beds. Even after best-checkpoint selection these three classes remain in Tier 3.
*Solutions:* (a) heavier backbone (ResNet-101, Swin-T) for finer discriminative features, (b) hard-example mining or focal loss in the classification head, (c) more (and more diverse) furniture training data. Really a data-side problem.

**3. Elongated-object localisation (`elongated_localisation`, 131 images).** *boat (0.78)* in particular suffers: boats are long, thin, often partially occluded by water or piers. The IoU=0.5 threshold is unforgiving for elongated objects because a small absolute pixel error along the long axis costs disproportionate IoU.
*Solutions:* (a) replace smooth-L1 with IoU / GIoU / DIoU regression loss, (b) add a Cascade R-CNN refinement stage to iteratively tighten the box, (c) longer training with stronger geometric augmentation.

**4. False positives in cluttered backgrounds (`background_fp`, 654 images, the single largest categorised mode).** Background regions with TV-like rectangles, leaves, or window panes occasionally trigger spurious confident detections (visible in the failure visualisation as red boxes with no green counterpart). 654 images affected makes this the biggest precision-side problem.
*Solutions:* (a) hard-negative mining, (b) Soft-NMS for better handling of overlapping detections, (c) raise the inference confidence threshold for deployment: mAP@0.5 is computed at `score>=0.05` to integrate over the full precision-recall curve, but a deployed system would normally pick a higher operating point (for example `score>=0.5`) which trades recall for precision and removes most of these spurious confident detections, (d) longer training so the classifier becomes more discriminative, though the val curve suggests we are already near-converged at epoch 5.

**5. Class confusion within visually similar groups (`animal_confusion`, 128 images).** Despite Tier-1/2 dominance, the animal cluster (cat 0.88, dog 0.87, cow 0.89, horse 0.90, sheep 0.86, bird 0.87) shows occasional within-cluster swaps, typically dog/cat at distance, or cow/horse/sheep in pasture scenes.
*Solutions:* (a) hard-example mining focused on confused pairs, (b) heavier backbone for fine-grained features, (c) class-balanced sampling so under-represented animal classes don't get dominated by `person`.

**6. Class imbalance.** `person` accounts for roughly a quarter of all annotations in VOC, and the model is correspondingly strong on it (0.89). Under-represented classes such as `sheep` and `cow` perform well in our run but with smaller per-image counts there is more variance.
*Solutions:* class-balanced batch sampling, class-aware loss re-weighting, or copy-paste augmentation for rare-class images.

**Summary: biggest expected gains for this model.**

1. **Higher input resolution + finer FPN level** would directly attack the small-object weakness that defines Tier 3 (pottedplant 0.64, chair 0.76, boat 0.78, bottle 0.81). This is the single change most likely to move mAP.
2. **Hard-negative mining or Soft-NMS** would attack the largest categorised failure mode (`background_fp`, 654 images) without retraining, a fast win for precision.
3. **GIoU / DIoU regression loss** would help elongated objects (boat) and would marginally improve every class.
4. **Heavier backbone (ResNet-101 or Swin-T)** would help disambiguate the visually similar furniture classes (chair vs sofa vs diningtable).
5. **Combine train + val for training** (5,011 images instead of 2,501) is the simplest immediate gain, around 2 to 3 mAP points expected, but requires giving up the validation signal we used for best-checkpoint selection in Task 1.

Stacking 1 to 4 on the same recipe would plausibly bring VOC 11-pt mAP@0.5 into the 0.88-0.90 range without redesigning the detector.

**All code implementation, experimental results, and written discussion must be completed within this notebook.**

**Please save your final work as an .ipynb file and submit.**

## **Marking rubric**
The Assignment 3 is totally 50 marks.

**Report Submission — 25 marks.**

Task 1: Object Detection Network Design — 17 marks

(Model implementation:5, Model / training pipeline customization and discussion :10,Performance:2)

Task 2: Detection Performance and Error Analysis — 8 marks

(Class-wise performance analysis:3, Error case visualization and discussion:3, Potential solutions:2)



**Oral Question — 25 marks**

Each group will be asked some questions in week 13.
